In [1]:
import google.auth
import numpy as np
import pandas as pd
import pygris 
import geopandas as gpd
from calitp_data_analysis import geography_utils
from calitp_data_analysis.sql import to_snakecase
from shared_utils import arcgis_query

In [2]:
from calitp_data_analysis import get_fs
fs = get_fs()

In [3]:

import os
from typing import List, Optional, Union
import pyarrow.dataset as ds
from google.cloud import storage

In [4]:
import google.auth
import pandas_gbq

credentials, project = google.auth.default()
from functools import cache

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas

In [5]:


@cache
def gcs_geopandas():
    return GCSGeoPandas()

In [6]:
@cache
def gcs_pandas():
    return GCSPandas()

In [7]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

## Load Data
### Census Block

In [8]:
def chunked(seq, size):
    """Yield successive chunks of length 'size' from a sequence."""
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

In [9]:
def load_ca_blocks(year:int, variables:str, county_code: str):
    """
    Load all census blocks for California (2020 PL94 decennial).
    Downloads one county at a time because the API does not allow
    state-level block requests.
    """
    df = pygris.blocks(
        state="06",     # California
        county=county_code,   
        year=2020,
        cache=True
       )

    # Reproject
    df = df.to_crs(geography_utils.CA_NAD83Albers_ft)

    # Buffer
    df["b250"] = df.buffer(250)
    df = df.drop(columns = ["geometry"])

    # Save locally first
    df.to_parquet("./chunked.parquet")

    # Save to GCS
    ca_counties = to_snakecase(pygris.counties(state="CA", year=year)[["NAME","COUNTYFP"]])
    file_name = ca_counties.loc[ca_counties.countyfp == county_code].name.iloc[0]
    df["county_name"] = file_name
    
    # Put the local file into the GCS bucket
    df.to_parquet("./chunked.parquet")
    fs.put("./chunked.parquet", f"gs://calitp-analytics-data/data-analyses/equity_index/census_blocks/census_blocks_county_{file_name}.parquet")
    return df

In [11]:
for county in ca_counties_list:
    block_gdf = load_ca_blocks(2020, "P1_001N", county)
    print(f"Done with {county}")

Using FIPS code '06' for input 'CA'
Done with 091
Using FIPS code '06' for input 'CA'
Done with 067
Using FIPS code '06' for input 'CA'
Done with 083
Using FIPS code '06' for input 'CA'
Done with 009
Using FIPS code '06' for input 'CA'
Done with 111
Using FIPS code '06' for input 'CA'
Done with 037
Using FIPS code '06' for input 'CA'
Done with 097
Using FIPS code '06' for input 'CA'
Done with 031
Using FIPS code '06' for input 'CA'
Done with 073
Using FIPS code '06' for input 'CA'
Done with 061
Using FIPS code '06' for input 'CA'
Done with 075
Using FIPS code '06' for input 'CA'
Done with 041
Using FIPS code '06' for input 'CA'
Done with 043
Using FIPS code '06' for input 'CA'
Done with 035
Using FIPS code '06' for input 'CA'
Done with 055
Using FIPS code '06' for input 'CA'
Done with 089
Using FIPS code '06' for input 'CA'
Done with 053
Using FIPS code '06' for input 'CA'
Done with 105
Using FIPS code '06' for input 'CA'
Done with 045
Using FIPS code '06' for input 'CA'
Done with 027


In [12]:
def _parse_gcs_path(gcs_path: str):
    """
    Split a GCS URL into (bucket, prefix) without leading 'gs://'.
    """
    if not gcs_path.startswith("gs://"):
        raise ValueError(f"Expected a 'gs://' path, got: {gcs_path}")
    no_scheme = gcs_path[5:]
    bucket, *rest = no_scheme.split("/", 1)
    prefix = rest[0] if rest else ""
    if prefix and not prefix.endswith("/"):
        prefix += "/"
    return bucket, prefix

In [13]:
def list_gcs_files(gcs_folder: str, extensions: Optional[List[str]] = None) -> list:
    """
    List all files in a GCS 'folder' (prefix). Optionally filter by extensions.
    Returns full 'gs://...' URIs.
    """
    bucket_name, prefix = _parse_gcs_path(gcs_folder)
    client = storage.Client()
    bucket = client.bucket(bucket_name)

    uris: List[str] = []
    for blob in client.list_blobs(bucket_name, prefix=prefix):
        # Skip "directory placeholders"
        name = blob.name
        if name.endswith("/"):
            continue
        if extensions:
            if not any(name.lower().endswith(ext.lower()) for ext in extensions):
                continue
        uris.append(f"gs://{bucket_name}/{name}")

    return sorted(uris)

In [14]:

try:
    import geopandas as gpd
    _HAS_GPD = True
except Exception:
    _HAS_GPD = False


In [15]:
def concat_gcs_folder(
    gcs_folder: str = "gs://calitp-analytics-data/data-analyses/equity_index/tims/",
    prefer_arrow_dataset: bool = True,
    file_types: Optional[List[str]] = None,
    geometry: bool = False,
    dtype_overrides: Optional[dict] = None,
    use_threads: bool = True,
) -> Union[pd.DataFrame, "gpd.GeoDataFrame"]:
    """
    Concatenate all files in a GCS folder into a single DataFrame.

    Parameters
    ----------
    gcs_folder : str
        GCS folder URL to read.
    prefer_arrow_dataset : bool
        If True and all files are parquet, use pyarrow.dataset for fast ingestion.
    file_types : List[str] or None
        Limit to these extensions (e.g., ["parquet","csv","feather","geojson"]).
        If None, auto-detect common formats.
    geometry : bool
        If True and reading GeoJSON, return a GeoDataFrame (requires geopandas).
    dtype_overrides : dict or None
        Dict of column dtypes to enforce after read (e.g., {"GEOID":"string"}).
    use_threads : bool
        Pass-through to pandas readers to enable multi-threaded parsing when supported.

    Returns
    -------
    DataFrame or GeoDataFrame
    """
    # Default formats we’ll support
    if file_types is None:
        file_types = ["parquet", "csv", "feather", "geojson", "json"]  # last two for geospatial or line-delimited JSON

    files = list_gcs_files(gcs_folder, extensions=[f".{ext}" for ext in file_types])

    if not files:
        raise FileNotFoundError(f"No files found under: {gcs_folder} with types {file_types}")

    # If all files are parquet and arrow dataset is preferred → use PA dataset
    all_parquet = all(f.lower().endswith(".parquet") for f in files)
    if prefer_arrow_dataset and all_parquet:
        # Arrow can read the entire folder as one dataset (partition-aware)
        dataset = ds.dataset(gcs_folder, format="parquet")
        table = dataset.to_table(use_threads=use_threads)
        df = table.to_pandas(types_mapper=pd.ArrowDtype)
        if dtype_overrides:
            df = df.astype(dtype_overrides, errors="ignore")
        return df

    # Otherwise, iterate by type and read via pandas / geopandas
    frames: List[Union[pd.DataFrame, "gpd.GeoDataFrame"]] = []

    for uri in files:
        lower = uri.lower()
        if lower.endswith(".parquet"):
            # pandas supports GCS via fsspec/gcsfs
            frames.append(pd.read_parquet(uri))
        elif lower.endswith(".feather"):
            frames.append(pd.read_feather(uri))
        elif lower.endswith(".csv"):
            frames.append(pd.read_csv(uri, low_memory=False))
        elif lower.endswith(".geojson") or (lower.endswith(".json") and "geo" in os.path.basename(uri).lower()):
            if not _HAS_GPD:
                raise ImportError("geopandas not installed—install it or set geometry=False.")
            gdf = gpd.read_file(uri)
            frames.append(gdf)
        else:
            # Skip unknown formats
            print(f"[concat_gcs_folder] Skipping unsupported file: {uri}")

    if not frames:
        raise FileNotFoundError(f"Found files, but none were readable with the allowed types: {file_types}")

    # Concatenate; if any GeoDataFrames present, upcast to GeoDataFrame
    if _HAS_GPD and any(isinstance(f, gpd.GeoDataFrame) for f in frames):
        df = pd.concat(frames, ignore_index=True)
        # Rebuild geometry column if lost during concat (rare)
        if "geometry" in df.columns and not isinstance(df, gpd.GeoDataFrame):
            df = gpd.GeoDataFrame(df, geometry="geometry", crs=frames[0].crs if hasattr(frames[0], "crs") else None)
    else:
        df = pd.concat(frames, ignore_index=True)

    if dtype_overrides:
        df = df.astype(dtype_overrides, errors="ignore")

    df = to_snakecase(df)
    return df

In [20]:

census_blocks_gdf = concat_gcs_folder(
    gcs_folder = "gs://calitp-analytics-data/data-analyses/equity_index/census_blocks",
    prefer_arrow_dataset =True,
    file_types = None,
    geometry = True,
    dtype_overrides = None,
    use_threads = True,
)


ArrowInvalid: google::cloud::Status(INVALID_ARGUMENT: Permanent error, with a last message of Could not create a OAuth2 access token to authenticate the request. The request was not sent, as such an access token is required to complete the request successfully. Learn more about Google Cloud authentication at https://cloud.google.com/docs/authentication. The underlying error message was: Unsupported credential type (external_account_authorized_user) when reading Application Default Credentials file from /home/jovyan/.config/gcloud/application_default_credentials.json. error_info={reason=INVALID_ARGUMENT, domain=gcloud-cpp, metadata={gcloud-cpp.retry.function=GetObjectMetadata, gcloud-cpp.source.function=LoadCredsFromString, gcloud-cpp.source.line=99, gcloud-cpp.source.filename=/opt/vcpkg/buildtrees/google-cloud-cpp/src/v2.37.0-dcffbfa290.clean/google/cloud/internal/oauth2_google_credentials.cc, gcloud-cpp.retry.reason=permanent-error, gcloud-cpp.retry.original-message=Could not create a

### Public Road Functional Classification
**Amanda** Need to fix: URL maxes out at 2000 rows when there are thousands more. 

In [ ]:
# https://caltrans-gis.dot.ca.gov/arcgis/rest/services/CHhighway/CRS_Functional_Classification/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson

In [ ]:
public_road_url = "https://caltrans-gis.dot.ca.gov/arcgis/rest/services/CHhighway/CRS_Functional_Classification/FeatureServer/0/"

In [ ]:
public_road_gdf = to_snakecase(gcs_geopandas().read_parquet("gs://calitp-analytics-data/data-analyses/shared_data/public_road_functional_classification.parquet"))

In [ ]:
public_road_gdf.f_system.unique()

In [ ]:
interstate_freeway = public_road_gdf.loc[public_road_gdf["f_system"].isin([1,2])]

In [ ]:
len(interstate_freeway), len(public_road_gdf)

In [ ]:
interstate_freeway = interstate_freeway.to_crs(geography_utils.CA_NAD83Albers_ft)

In [ ]:
interstate_freeway["b50"] = interstate_freeway.geometry.buffer(50)

In [ ]:
interstate_freeway = interstate_freeway.drop(columns = ["geometry"])

In [ ]:
interstate_freeway = interstate_freeway.set_geometry("b50")

### TIMS Data

In [ ]:

tims_gdf = concat_gcs_folder(
    gcs_folder="gs://calitp-analytics-data/data-analyses/equity_index/tims/",
    prefer_arrow_dataset=True
)


In [ ]:
tims_gdf = gpd.GeoDataFrame(
    tims_df, geometry=gpd.points_from_xy(tims_gdf.point_x, tims_gdf.point_y), crs=geography_utils.WGS84 
).to_crs(geography_utils.CA_NAD83Albers_ft)

In [ ]:
tims_gdf.case_id.nunique()

In [ ]:
tims_gdf.shape

## Overlay TIMS with Public Road Functional Classification data for crashes we don't want. 
* Filter them out

In [ ]:
tims_public_road = (
        tims_gdf.sjoin(interstate_freeway, how="inner", predicate="intersects")
        .reset_index(drop=True)
        .drop(columns=["index_right"])
    )

In [ ]:
len(tims_public_road)

In [ ]:
tims_public_road.head(1)

In [ ]:
crashes_to_delete = list(tims_public_road.case_id.unique())

In [ ]:
tims_gdf2 = tims_gdf.loc[~tims_gdf.case_id.isin(crashes_to_delete)]

In [ ]:
tims_gdf2.shape